In [20]:
import numpy as np
from ase.io import read, write
from ase.optimize import BFGS
from xtb.ase.calculator import XTB   # recommended for pre-opt

from rdkit import Chem
from rdkit.Chem import AllChem

from pyscf import gto, scf
from pyscf.lib import logger

# === Correct modern ASF imports ===
from asf.wrapper import find_from_mol, find_from_scf
from asf.utility import pictures_Jmol

print("=== Full Workflow: 1CA2.pdb → Realistic Ligand Placement → ASF → PennyLane ===")

# ===================================================================
# 1. Extract Zn-centered cluster from 1CA2.pdb (6.5 Å is good for testing)
# ===================================================================
atoms = read("/home/loharkar/QuEnAIS-quantum-embedding/data/raw/1AH7.pdb")

zn_idx = [i for i, a in enumerate(atoms) if a.symbol == "Zn"][0]
zn_pos = atoms[zn_idx].position.copy()

sel = [i for i, a in enumerate(atoms) 
       if np.linalg.norm(a.position - zn_pos) < 3.0]

cluster = atoms[sel]


#from ase.build import add_hydrogens
#cluster = add_hydrogens(cluster)
write("cluster.xyz", cluster)
print(f"Cluster extracted: {len(cluster)} atoms")
print("STEP 1 DONE")


=== Full Workflow: 1CA2.pdb → Realistic Ligand Placement → ASF → PennyLane ===
Cluster extracted: 9 atoms
STEP 1 DONE


In [21]:
# ===================================================================
# 2. Prepare ligands + realistic placement using Zn-water vector
# ===================================================================
ligand_name = "L1"   # Change later to L2, L3, or any new ligand

# Generate ligand (same as before)
mol_rd = Chem.MolFromSmiles("c1ncc[nH]1")   # L1 = imidazole
mol_rd = Chem.AddHs(mol_rd)
AllChem.EmbedMolecule(mol_rd, randomSeed=42)
AllChem.UFFOptimizeMolecule(mol_rd)
Chem.MolToXYZFile(mol_rd, f"{ligand_name}.xyz")

# Load cluster and ligand
cluster = read("cluster.xyz")
lig = read(f"{ligand_name}.xyz")

zn_idx = [i for i, a in enumerate(cluster) if a.symbol == "Zn"][0]
zn_pos = cluster[zn_idx].position.copy()

# Find the closest water to Zn (this is the most physical direction)
water_candidates = [i for i, a in enumerate(cluster) 
                    if a.symbol == "O" and 1.8 < np.linalg.norm(a.position - zn_pos) < 2.8]

if water_candidates:
    water_idx = min(water_candidates, key=lambda i: np.linalg.norm(cluster[i].position - zn_pos))
    direction = cluster[water_idx].position - zn_pos
    direction /= np.linalg.norm(direction)
    print(f"Using real Zn-water vector for placement (distance = {np.linalg.norm(cluster[water_idx].position - zn_pos):.2f} Å)")
else:
    direction = np.array([0.0, 0.0, 1.0])

# Place coordinating atom (N for imidazole) at exactly 2.0 Å
n_idx = [i for i, a in enumerate(lig) if a.symbol == "N"][0]
lig.positions -= lig.positions[n_idx]
lig.positions += zn_pos + 2.0 * direction

# Final complex
combined = cluster + lig
combined.set_pbc(False)
combined.set_cell([0, 0, 0])
combined.center()

write(f"{ligand_name}_complex_initial.xyz", combined)
print(f"\nSTEP 2 DONE → {ligand_name} placed at Zn–N = 2.0 Å using water vector")
print(f"File saved: {ligand_name}_complex_initial.xyz")

Using real Zn-water vector for placement (distance = 2.19 Å)

STEP 2 DONE → L1 placed at Zn–N = 2.0 Å using water vector
File saved: L1_complex_initial.xyz


In [22]:
from ase.optimize import BFGS
from xtb.ase.calculator import XTB

combined = read(f"{ligand_name}_complex_initial.xyz")

combined.calc = XTB(method="GFN2-xTB")

opt = BFGS(combined)
opt.run(fmax=0.15)

write(f"{ligand_name}_opt.xyz", combined)
print("STEP 3 DONE → Geometry optimized")

      Step     Time          Energy          fmax
BFGS:    0 17:03:00      496.217885    12806.175270
BFGS:    1 17:03:12     -946.607311      606.929839
BFGS:    2 17:03:15     -963.370371      547.080923
BFGS:    3 17:03:18     -985.993809      408.482434
BFGS:    4 17:03:21     -998.236922      243.987891
BFGS:    5 17:03:24    -1020.334941       92.771759
BFGS:    6 17:03:27    -1031.793144       23.221175
BFGS:    7 17:03:31    -1035.685383       20.122714
BFGS:    8 17:03:34    -1039.672613        7.280492
BFGS:    9 17:03:36    -1043.775289        5.271680
BFGS:   10 17:03:39    -1043.873968       15.750517
BFGS:   11 17:03:41    -1045.377012        5.339404
BFGS:   12 17:03:44    -1046.071961        5.132995
BFGS:   13 17:03:47    -1048.091974        6.941654
BFGS:   14 17:03:49    -1050.092654        9.254541
BFGS:   15 17:03:51    -1051.462627        7.010527
BFGS:   16 17:03:53    -1052.591145        5.729036
BFGS:   17 17:03:56    -1053.614993        9.353859
BFGS:   18 17:

In [24]:
from pyscf import gto

atoms = read(f"{ligand_name}_opt.xyz")

atom_str = ""
for a in atoms:
    atom_str += f"{a.symbol} {a.position[0]} {a.position[1]} {a.position[2]}\n"

mol = gto.Mole()
mol.atom = atom_str
mol.basis = "def2-svp"   # good starting point     
mol.charge = 0
mol.spin = 0
mol.build()
print("Number of electrons:", mol.nelectron)
print("STEP 4 DONE → PySCF molecule built")


Number of electrons: 122
STEP 4 DONE → PySCF molecule built


In [27]:
mf = scf.RKS(mol)
mf.verbose = 4
mf.xc = 'b3lyp'
mf.conv_tol = 1e-6
mf.conv_tol_grad = 1e-3      # looser gradient tolerance for difficult cases
mf.max_cycle = 50

print("Starting robust UKS calculation...")
mf.kernel()


Starting robust UKS calculation...


******** <class 'pyscf.dft.rks.RKS'> ********
method = RKS
initial guess = minao
damping factor = 0
level_shift factor = 0
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-06
SCF conv_tol_grad = 0.001
SCF max_cycles = 50
direct_scf = True
direct_scf_tol = 1e-13
chkfile to save SCF result = /tmp/tmpz7uz7vjd
max_memory 4000 MB (current use 3526 MB)
XC library pyscf.dft.libxc version 7.0.0
    S. Lehtola, C. Steigemann, M. J.T. Oliveira, and M. A.L. Marques.,  SoftwareX 7, 1–5 (2018)
XC functionals = b3lyp
    P. J. Stephens, F. J. Devlin, C. F. Chabalowski, and M. J. Frisch.,  J. Phys. Chem. 98, 11623 (1994)
small_rho_cutoff = 1e-07
Initial guess from minao.
init E= -2456.28476428508
  HOMO = -0.199039003248887  LUMO = -0.138064675919027
cycle= 1 E= -2453.81528030689  delta_E= 2.47  |g|= 1.12  |ddm|= 5.38
  HOMO = -0.134485218605358  LUMO = -0.112111321138181
cycle= 2 E= -2452.78823494242  delta

np.float64(-2454.05600445033)

In [26]:
from asf.wrapper import find_from_scf

asf_result = find_from_scf(
    mf,
    max_norb=6,
    verbose=True  # freeze deep core orbitals
)


--------------------------------------------------------------------------------
Calculating MP2 natural orbitals
--------------------------------------------------------------------------------


-> Selected initial orbital window of 30 electrons in 30 MP2 natural orbitals.

--------------------------------------------------------------------------------
Running calculation
--------------------------------------------------------------------------------

spin = 0, number of roots = 1
total number of roots calculated = 1


ERROR:  /home/loharkar/QuEnAIS-quantum-embedding/quenais-env2/bin/block2main dmrg.conf > dmrg.out 2>&1


********************************** INPUT START **********************************
nelec                                                           30
spin                                                             0
twodot_to_onedot                                                14
orbitals                                                   FCIDUMP
maxiter        

ERROR:  /home/loharkar/QuEnAIS-quantum-embedding/quenais-env2/bin/block2main dmrg.conf > dmrg.out 2>&1


CalledProcessError: Command ' /home/loharkar/QuEnAIS-quantum-embedding/quenais-env2/bin/block2main dmrg.conf > dmrg.out 2>&1' returned non-zero exit status 2.

In [ ]:
active_space = find_from_scf(mf)            # reorder MOs for direct use in CASSCF/CASCI



--------------------------------------------------------------------------------
Calculating MP2 natural orbitals
--------------------------------------------------------------------------------



KeyboardInterrupt: 

In [ ]:
from pyscf import scf

mf = scf.RHF(mol)   # or UKS if open-shell
mf.verbose = 4
mf.kernel()

print("SCF Energy:", mf.e_tot)



******** <class 'pyscf.scf.rohf.ROHF'> ********
method = ROHF
initial guess = minao
damping factor = 0
level_shift factor = 0
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-09
SCF conv_tol_grad = None
SCF max_cycles = 50
direct_scf = True
direct_scf_tol = 1e-13
chkfile to save SCF result = /tmp/tmpf_ole90u
max_memory 4000 MB (current use 1973 MB)
num. doubly occ = 123  num. singly occ = 1
Set gradient conv threshold to 3.16228e-05
init E= -4960.13696333363
  HOMO = -0.111738084375348  LUMO = -0.0959439650152499
cycle= 1 E= -4952.1056621992  delta_E= 8.03  |g|= 1.89  |ddm|= 5.33
  HOMO = -0.195847941619773  LUMO = -0.13711741186324
cycle= 2 E= -4950.18918840286  delta_E= 1.92  |g|= 2.68  |ddm|= 3.41
  HOMO = -0.195887000382177  LUMO = -0.116601723867849
cycle= 3 E= -4952.66607308993  delta_E= -2.48  |g|= 1.18  |ddm|= 3.03
  HOMO = -0.120401806053021  LUMO = -0.112607256927479
cycle= 4 E= -4953.20202825637  delta_E= -0.536  |g|

In [ ]:
from pyscf import scf
mf = scf.RKS(mol)
mf.verbose = 4
mf.xc = "PBE"   # or B3LYP
mf.kernel()
print("SCF Energy:", mf.e_tot)



******** <class 'pyscf.dft.roks.ROKS'> ********
method = ROKS
initial guess = minao
damping factor = 0
level_shift factor = 0
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-09
SCF conv_tol_grad = None
SCF max_cycles = 50
direct_scf = True
direct_scf_tol = 1e-13
chkfile to save SCF result = /tmp/tmpatjtnna0
max_memory 4000 MB (current use 1795 MB)
num. doubly occ = 123  num. singly occ = 1
XC library pyscf.dft.libxc version 7.0.0
    S. Lehtola, C. Steigemann, M. J.T. Oliveira, and M. A.L. Marques.,  SoftwareX 7, 1–5 (2018)
XC functionals = PBE
    J. P. Perdew, K. Burke, and M. Ernzerhof.,  Phys. Rev. Lett. 77, 3865 (1996)
    J. P. Perdew, K. Burke, and M. Ernzerhof.,  Phys. Rev. Lett. 78, 1396 (1997)
    J. P. Perdew, K. Burke, and M. Ernzerhof.,  Phys. Rev. Lett. 77, 3865 (1996)
    J. P. Perdew, K. Burke, and M. Ernzerhof.,  Phys. Rev. Lett. 78, 1396 (1997)
small_rho_cutoff = 1e-07
Set gradient conv threshold to 3.16228e-

KeyboardInterrupt: 

In [ ]:
active_space = find_from_scf(
    mf,
    
    entropy_threshold=0.15,      # tune: 0.1–0.25 works well for metal-ligand systems
    states=1,                    # ground state
    sort_mos=True                # reorder MOs for direct use in CASSCF/CASCI
)


--------------------------------------------------------------------------------
Calculating MP2 natural orbitals
--------------------------------------------------------------------------------



AttributeError: 'NoneType' object has no attribute 'shape'